In [ ]:
# =============================================================================
# CELL 1: SETUP & DATA LOADING — ITALY
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
import optuna
import time
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("=" * 70)
print("ITALY COVID-19 FORECASTING — FULL PIPELINE (v9)")
print("=" * 70)

owid_df = pd.read_csv('../data/filtered_compact.csv')
owid_df['date'] = pd.to_datetime(owid_df['date'])

it = owid_df[owid_df['country'] == 'Italy'].copy()
it = it.sort_values('date').reset_index(drop=True)
it = it[it['date'] <= '2022-06-30'].copy()

print(f"\nItaly data: {len(it)} rows, {it['date'].min().date()} to {it['date'].max().date()}")

# Check key columns
for col in ['new_deaths', 'new_deaths_smoothed', 'new_cases', 'new_cases_smoothed',
            'icu_patients', 'hosp_patients', 'weekly_hosp_admissions',
            'reproduction_rate', 'positive_rate', 'stringency_index',
            'people_fully_vaccinated', 'total_boosters', 'people_vaccinated']:
    if col in it.columns:
        n = it[col].notna().sum()
        print(f"  {col:<40} {n:>4} non-null ({n/len(it)*100:.0f}%)")
    else:
        print(f"  {col:<40} MISSING")

In [ ]:
# =============================================================================
# CELL 2: CLEAN DATA PREP — ITALY
# =============================================================================
print("\n" + "=" * 70)
print("DATA PREPARATION")
print("=" * 70)

# Set date as index
it = it.set_index('date')
it.index = pd.to_datetime(it.index)

# Full daily index (no gaps)
full_idx = pd.date_range(it.index.min(), it.index.max(), freq='D')
it = it.reindex(full_idx)
it.index.name = 'date'

# Count columns: fill missing with 0
for col in ['new_deaths', 'new_cases', 'new_tests_smoothed']:
    if col in it.columns:
        it[col] = it[col].fillna(0)

# Smoothed columns
for col in ['new_deaths_smoothed', 'new_cases_smoothed']:
    if col in it.columns:
        it[col] = it[col].ffill().bfill()
    else:
        raw = col.replace('_smoothed', '')
        if raw in it.columns:
            it[col] = it[raw].rolling(14, min_periods=1).mean()

# Rate columns
for col in ['reproduction_rate', 'icu_patients', 'hosp_patients',
            'positive_rate', 'weekly_hosp_admissions', 'stringency_index']:
    if col in it.columns:
        it[col] = it[col].ffill().bfill()

# Vaccination
for col in ['people_fully_vaccinated', 'total_boosters', 'people_vaccinated']:
    if col in it.columns:
        it[col] = it[col].ffill().fillna(0)

print(f"After daily reindex: {len(it)} rows")
print(f"Date range: {it.index.min().date()} to {it.index.max().date()}")

for col in ['new_deaths_smoothed', 'new_cases_smoothed', 'icu_patients',
            'hosp_patients', 'reproduction_rate', 'positive_rate',
            'weekly_hosp_admissions', 'people_fully_vaccinated', 'total_boosters']:
    if col in it.columns:
        n = it[col].notna().sum()
        print(f"  {col:<40} {n:>4} non-null ({n/len(it)*100:.0f}%)")

# Quick check of Jan 2022 target ranges
jan22 = it[(it.index >= '2022-01-01') & (it.index <= '2022-01-31')]
print(f"\nJan 2022 Deaths: mean={jan22['new_deaths_smoothed'].mean():.1f}, "
      f"range=[{jan22['new_deaths_smoothed'].min():.1f}, {jan22['new_deaths_smoothed'].max():.1f}]")
print(f"Jan 2022 Cases:  mean={jan22['new_cases_smoothed'].mean():.0f}, "
      f"range=[{jan22['new_cases_smoothed'].min():.0f}, {jan22['new_cases_smoothed'].max():.0f}]")

In [ ]:
# =============================================================================
# CELL 3: FEATURE ENGINEERING — ITALY (14-day lags, matching Germany)
# =============================================================================
print("\n" + "=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

def build_features_italy(df, target_type='deaths'):
    """
    Build features with 14-day minimum lag (no data leakage).
    Matches Germany's feature engineering approach.
    """
    data = df.copy()
    pop = 59.6e6

    if target_type == 'deaths':
        data['target'] = data['new_deaths_smoothed'].clip(lower=0)
    else:
        data['target'] = data['new_cases_smoothed'].clip(lower=0)

    features = []

    # --- 1. Core target lags (14-day minimum) ---
    for lag in [14, 21, 28]:
        col = f'lag_{lag}'
        data[col] = data['target'].shift(lag)
        features.append(col)

    # --- 2. Rolling statistics (shifted by 14 to avoid leakage) ---
    for window in [ 14]:
        col_mean = f'roll_mean_{window}'
        col_std  = f'roll_std_{window}'
        data[col_mean] = data['target'].shift(14).rolling(window, min_periods=1).mean()
        data[col_std]  = data['target'].shift(14).rolling(window, min_periods=1).std().fillna(0)
        features.extend([col_mean, col_std])

    # --- 3. Momentum & trend ---
    data['momentum_14d'] = data['target'].shift(14) / data['target'].shift(21).replace(0, 1)
    data['slope_14d']  = (data['target'].shift(14) - data['target'].shift(28)) / 14
    features.extend(['momentum_7', 'slope_14d'])

    # --- 4. Log-transform & EMA ---
    data['log_lag_14'] = np.log1p(data['target'].shift(14))
    data['ema_14']     = data['target'].ewm(span=7).mean().shift(14)
    features.extend(['log_lag_14', 'ema_14'])

    # --- 5. External indicators (lagged by 14) ---
    if 'hosp_patients' in data.columns:
        data['hosp_lag_14'] = data['hosp_patients'].shift(14)
        features.append('hosp_lag_14')

    if 'icu_patients' in data.columns:
        data['icu_lag_14'] = data['icu_patients'].shift(14)
        features.append('icu_lag_14')

    if 'reproduction_rate' in data.columns:
        data['rt_lag_14'] = data['reproduction_rate'].shift(14)
        features.append('rt_lag_14')

    if 'positive_rate' in data.columns:
        data['posrate_lag_14'] = data['positive_rate'].shift(14)
        features.append('posrate_lag_14')

    if 'stringency_index' in data.columns:
        data['stringency_lag_14'] = data['stringency_index'].shift(14)
        features.append('stringency_lag_14')

    # --- 6. Vaccination (cumulative, no lag needed) ---
    if 'people_fully_vaccinated' in data.columns:
        data['vacc_rate'] = data['people_fully_vaccinated'] / pop
        features.append('vacc_rate')

    if 'total_boosters' in data.columns:
        data['booster_rate'] = data['total_boosters'] / pop
        features.append('booster_rate')

    # --- 7. Cross-signal (deaths↔cases relationship) ---
    if target_type == 'deaths' and 'new_cases_smoothed' in data.columns:
        data['cases_lag_14'] = data['new_cases_smoothed'].shift(14)
        data['cases_lag_21'] = data['new_cases_smoothed'].shift(21)
        features.extend(['cases_lag_14', 'cases_lag_21'])

    if target_type == 'cases' and 'new_deaths_smoothed' in data.columns:
        data['deaths_lag_14'] = data['new_deaths_smoothed'].shift(14)
        features.append('deaths_lag_14')

    # --- 8. Day of week ---
    data['dow'] = data.index.dayofweek
    data['is_weekend'] = (data['dow'] >= 5).astype(int)
    features.extend(['dow', 'is_weekend'])

    # Reset index to have 'date' as column (matching Germany format)
    data = data.reset_index()
    data = data.rename(columns={'index': 'date'} if 'index' in data.columns else {})
    if 'date' not in data.columns and data.index.name == 'date':
        data = data.reset_index()

    # Keep only target + features + date
    keep_cols = ['date', 'target'] + [f for f in features if f in data.columns]
    data = data[keep_cols].dropna()

    # Filter: start after enough lag data exists
    data = data[data['date'] >= '2020-04-01'].copy()

    features = [f for f in features if f in data.columns]
    print(f"  {target_type.title()}: {len(data)} rows, {len(features)} features")

    return data, features


df_deaths, feats_d, = build_features_italy(it, 'deaths')
df_cases,  feats_c  = build_features_italy(it, 'cases')

print(f"\n  Deaths features: {feats_d}")
print(f"  Cases features:  {feats_c}")

In [ ]:
# =============================================================================
# CELL 4: TEMPORAL SPLIT — MATCHING GERMANY
# =============================================================================
print("\n" + "=" * 70)
print("TEMPORAL SPLIT (matching Germany)")
print("=" * 70)

def temporal_split_aligned(data, label=''):
    """
    Same split logic as Germany:
      Train: everything before 2021-01-01
      Val:   January 2021 (backtest period)
      Test:  January 2022 (forecast period)
    """
    train = data[data['date'] <  '2021-01-01'].copy()
    val   = data[(data['date'] >= '2021-01-01') & (data['date'] <= '2021-01-31')].copy()
    test  = data[(data['date'] >= '2022-01-01') & (data['date'] <= '2022-01-31')].copy()

    # Also create train+val for retraining
    train_val = data[data['date'] < '2022-01-01'].copy()

    print(f"\n  {label} Split:")
    print(f"    Train:     {len(train):>4} rows  ({train['date'].min().date()} to {train['date'].max().date()})")
    print(f"    Val:       {len(val):>4} rows  ({val['date'].min().date()} to {val['date'].max().date()})")
    print(f"    Test:      {len(test):>4} rows  ({test['date'].min().date()} to {test['date'].max().date()})")
    print(f"    Train+Val: {len(train_val):>4} rows")

    if len(test) > 0 and len(train) > 0:
        print(f"    Target — Train mean: {train['target'].mean():.1f}, "
              f"Test mean: {test['target'].mean():.1f}, "
              f"OOD ratio: {test['target'].mean()/train['target'].mean():.2f}x")

    return {'train': train, 'val': val, 'test': test, 'train_val': train_val}


split_d = temporal_split_aligned(df_deaths, label='Deaths')
split_c = temporal_split_aligned(df_cases,  label='Cases')

In [ ]:
# =============================================================================
# CELL 5: DEATHS MODEL TRAINING (v9 OPTIMIZED)
# =============================================================================
print("\n" + "=" * 70)
print("MODEL TRAINING — ITALY DEATHS (v9)")
print("=" * 70)

cell_start = time.time()

train_d     = split_d['train']
val_d       = split_d['val']
test_d      = split_d['test']
train_val_d = split_d['train_val']

results_d = {}

all_d_feats = [f for f in feats_d if f in train_d.columns]

print(f"\n  Train: n={len(train_d)}, max={train_d['target'].max():.0f}, mean={train_d['target'].mean():.0f}")
print(f"  Val:   n={len(val_d)}, max={val_d['target'].max():.0f}, mean={val_d['target'].mean():.0f}")
print(f"  Test:  n={len(test_d)}, max={test_d['target'].max():.0f}, mean={test_d['target'].mean():.0f}")
print(f"  Features: {len(all_d_feats)}")


# ==========================================================================
# HELPERS
# ==========================================================================
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true < 1, 1, y_true))) * 100
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}


def store_results(results_dict, model_name, model, splits, feat_cols,
                  scaler=None, log_target=False):
    """Predict on each split, store predictions + metrics."""
    results_dict[model_name] = {}
    for sname, sdf in splits:
        X = sdf[feat_cols].values
        if scaler is not None:
            X = scaler.transform(X)
        raw = model.predict(X)
        pred = np.clip(np.expm1(raw), 0, None) if log_target else np.clip(raw, 0, None)
        actual = sdf['target'].values
        metrics = evaluate(f'{model_name} {sname}', actual, pred)
        results_dict[model_name][sname.lower()] = {
            'predictions': pred, 'actual': actual,
            'dates': sdf['date'].values, **metrics
        }
    t = results_dict[model_name]['test']
    print(f"    {model_name:<45} Test MAE={t['mae']:>8.2f}  R²={t['r2']:>7.3f}  MAPE={t['mape']:>5.1f}%")


def batch_eval(results_dict, model_name, model, X_train, y_train,
               X_matrices, y_actuals, date_dicts, log_target=True):
    """Fit + predict on pre-transformed matrices."""
    model.fit(X_train, y_train)
    results_dict[model_name] = {}
    for sname in ['train', 'val', 'test']:
        raw = model.predict(X_matrices[sname])
        pred = np.clip(np.expm1(raw), 0, None) if log_target else np.clip(raw, 0, None)
        actual = y_actuals[sname]
        if len(pred) != len(actual):
            continue
        metrics = evaluate(f'{model_name} {sname}', actual, pred)
        results_dict[model_name][sname] = {
            'predictions': pred, 'actual': actual,
            'dates': date_dicts[sname], **metrics
        }
    if 'test' in results_dict[model_name]:
        t = results_dict[model_name]['test']
        v = results_dict.get(model_name, {}).get('val', {})
        v_mae = v.get('mae', np.nan) if v else np.nan
        print(f"    {model_name:<45} Val={v_mae:>8.2f}  Test={t['mae']:>8.2f}  R²={t['r2']:>7.3f}")


ALL_SPLITS_D = [('Train', train_d), ('Val', val_d), ('Test', test_d)]


# ==========================================================================
# PRE-COMPUTE
# ==========================================================================
t0 = time.time()

scaler_t_d  = RobustScaler().fit(train_d[all_d_feats])
scaler_tv_d = RobustScaler().fit(train_val_d[all_d_feats])

X_t_d  = scaler_t_d.transform(train_d[all_d_feats])
X_tv_d = scaler_tv_d.transform(train_val_d[all_d_feats])

y_t_d      = train_d['target'].values
y_tv_d     = train_val_d['target'].values
y_t_log_d  = np.log1p(y_t_d)
y_tv_log_d = np.log1p(y_tv_d)

X_val_t_d  = scaler_t_d.transform(val_d[all_d_feats])
X_test_t_d = scaler_t_d.transform(test_d[all_d_feats])
X_val_tv_d  = scaler_tv_d.transform(val_d[all_d_feats])
X_test_tv_d = scaler_tv_d.transform(test_d[all_d_feats])

X_mats_t_d  = {'train': X_t_d,  'val': X_val_t_d,  'test': X_test_t_d}
X_mats_tv_d = {'train': X_tv_d, 'val': X_val_tv_d, 'test': X_test_tv_d}

y_act_t_d  = {'train': y_t_d,  'val': val_d['target'].values, 'test': test_d['target'].values}
y_act_tv_d = {'train': y_tv_d, 'val': val_d['target'].values, 'test': test_d['target'].values}

dates_t_d  = {'train': train_d['date'].values, 'val': val_d['date'].values, 'test': test_d['date'].values}
dates_tv_d = {'train': train_val_d['date'].values, 'val': val_d['date'].values, 'test': test_d['date'].values}

print(f"  Pre-computed in {time.time()-t0:.1f}s")
print(f"  Train: {X_t_d.shape}, Train+Val: {X_tv_d.shape}")


# ==========================================================================
# SECTION 1: TREE-BASED MODELS
# ==========================================================================
print(f"\n{'='*60}")
print("  SECTION 1: TREE-BASED MODELS")
print(f"{'='*60}")
t0 = time.time()

# [1] LightGBM Huber log-target
print("\n  [1] LightGBM-Huber-Log:")
lgb_d = lgb.LGBMRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_samples=15,
    objective='huber', huber_delta=1.0,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_d.fit(train_d[all_d_feats], y_t_log_d,
          eval_set=[(val_d[all_d_feats], np.log1p(val_d['target']))],
          callbacks=[lgb.early_stopping(50, verbose=False)])
store_results(results_d, 'LightGBM-Huber-Log', lgb_d, ALL_SPLITS_D,
              feat_cols=all_d_feats, log_target=True)

# [2] LGB Huber Log TV
print("\n  [2] LGB-Huber-Log-TV:")
lgb_tv_d = lgb.LGBMRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_samples=15,
    objective='huber', huber_delta=1.0,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_tv_d.fit(train_val_d[all_d_feats], y_tv_log_d)
store_results(results_d, 'LGB-Huber-Log-TV', lgb_tv_d, ALL_SPLITS_D,
              feat_cols=all_d_feats, log_target=True)

# [3] XGBoost Log
print("\n  [3] XGBoost-Log:")
xgb_d = XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=15,
    gamma=0.5, random_state=42, n_jobs=-1, early_stopping_rounds=50
)
xgb_d.fit(train_d[all_d_feats], y_t_log_d,
          eval_set=[(val_d[all_d_feats], np.log1p(val_d['target']))], verbose=False)
store_results(results_d, 'XGBoost-Log', xgb_d, ALL_SPLITS_D,
              feat_cols=all_d_feats, log_target=True)

# [4] XGBoost Log TV
print("\n  [4] XGB-Log-TV:")
xgb_tv_d = XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=15,
    gamma=0.5, random_state=42, n_jobs=-1
)
xgb_tv_d.fit(train_val_d[all_d_feats], y_tv_log_d, verbose=False)
store_results(results_d, 'XGB-Log-TV', xgb_tv_d, ALL_SPLITS_D,
              feat_cols=all_d_feats, log_target=True)

# [5] XGBoost raw
print("\n  [5] XGBoost:")
xgb_raw_d = XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=15,
    gamma=0.5, random_state=42, n_jobs=-1, early_stopping_rounds=50
)
xgb_raw_d.fit(train_d[all_d_feats], y_t_d,
              eval_set=[(val_d[all_d_feats], val_d['target'])], verbose=False)
store_results(results_d, 'XGBoost', xgb_raw_d, ALL_SPLITS_D, feat_cols=all_d_feats)

# [6] LightGBM MAE
print("\n  [6] LightGBM-MAE:")
lgb_mae_d = lgb.LGBMRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_samples=15,
    objective='mae', random_state=42, n_jobs=-1, verbose=-1
)
lgb_mae_d.fit(train_d[all_d_feats], y_t_d,
              eval_set=[(val_d[all_d_feats], val_d['target'])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
store_results(results_d, 'LightGBM-MAE', lgb_mae_d, ALL_SPLITS_D, feat_cols=all_d_feats)

# [7] GB Huber
print("\n  [7] GB-Huber:")
gb_d = GradientBoostingRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.7, min_samples_leaf=15,
    loss='huber', alpha=0.9, random_state=42
)
gb_d.fit(train_d[all_d_feats], y_t_d)
store_results(results_d, 'GB-Huber', gb_d, ALL_SPLITS_D, feat_cols=all_d_feats)

print(f"\n  Tree models: {time.time()-t0:.1f}s")


# ==========================================================================
# SECTION 2: OPTUNA-TUNED LGB
# ==========================================================================
print(f"\n{'='*60}")
print("  SECTION 2: OPTUNA-TUNED LGB (200 trials)")
print(f"{'='*60}")
t0 = time.time()

def objective_lgb_d(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 20.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'huber_delta': trial.suggest_float('huber_delta', 0.5, 2.0),
    }
    model = lgb.LGBMRegressor(**params, objective='huber',
                               random_state=42, n_jobs=-1, verbose=-1)
    model.fit(train_d[all_d_feats], y_t_log_d)
    pred = np.clip(np.expm1(model.predict(val_d[all_d_feats])), 0, None)
    return mean_absolute_error(val_d['target'].values, pred)

study_lgb = optuna.create_study(direction="minimize")
study_lgb.optimize(objective_lgb_d, n_trials=200)
bp = study_lgb.best_params
hd = bp.pop('huber_delta')
print(f"    Best Val MAE: {study_lgb.best_value:.2f}")

lgb_opt_tv = lgb.LGBMRegressor(**bp, objective='huber', huber_delta=hd,
                                random_state=42, n_jobs=-1, verbose=-1)
lgb_opt_tv.fit(train_val_d[all_d_feats], y_tv_log_d)
store_results(results_d, 'LGB-Opt-TV', lgb_opt_tv, ALL_SPLITS_D,
              feat_cols=all_d_feats, log_target=True)

lgb_opt_t = lgb.LGBMRegressor(**bp, objective='huber', huber_delta=hd,
                               random_state=42, n_jobs=-1, verbose=-1)
lgb_opt_t.fit(train_d[all_d_feats], y_t_log_d)
store_results(results_d, 'LGB-Opt-T', lgb_opt_t, ALL_SPLITS_D,
              feat_cols=all_d_feats, log_target=True)

print(f"  Optuna: {time.time()-t0:.1f}s")


# ==========================================================================
# SECTION 3: LINEAR MODELS
# ==========================================================================
print(f"\n{'='*60}")
print("  SECTION 3: LINEAR MODELS")
print(f"{'='*60}")
t0 = time.time()

# Huber (Optuna)
print("\n  [8] Huber (Optuna, 100 trials):")
def objective_hub_d(trial):
    a = trial.suggest_float("alpha", 1e-5, 10.0, log=True)
    e = trial.suggest_float("epsilon", 1.1, 2.0)
    m = HuberRegressor(alpha=a, epsilon=e, max_iter=2000)
    m.fit(X_t_d, y_t_d)
    return mean_absolute_error(val_d['target'], m.predict(X_val_t_d))

study_h = optuna.create_study(direction="minimize")
study_h.optimize(objective_hub_d, n_trials=100)
print(f"    Best: {study_h.best_params}, Val MAE={study_h.best_value:.2f}")

hub_d = HuberRegressor(**study_h.best_params, max_iter=2000)
hub_d.fit(X_t_d, y_t_d)
store_results(results_d, 'Huber', hub_d, ALL_SPLITS_D,
              feat_cols=all_d_feats, scaler=scaler_t_d)

# Ridge
print("\n  [9] Ridge:")
for alpha in [1.0, 5.0, 10.0]:
    m = Ridge(alpha=alpha)
    m.fit(X_t_d, y_t_d)
    store_results(results_d, f'Ridge_a{alpha}', m, ALL_SPLITS_D,
                  feat_cols=all_d_feats, scaler=scaler_t_d)

# Ridge log-target
print("\n  [10] Ridge-Log:")
for alpha in [1.0, 5.0, 10.0]:
    batch_eval(results_d, f'Ridge-Log_a{alpha}_T', Ridge(alpha=alpha),
               X_t_d, y_t_log_d, X_mats_t_d, y_act_t_d, dates_t_d)

# ElasticNet raw (previous Italy winner)
print("\n  [11] ElasticNet (raw):")
for alpha in [0.5, 1.0, 2.0]:
    for l1 in [0.3, 0.5, 0.7, 0.9]:
        m = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=10000)
        m.fit(train_d[all_d_feats], y_t_d)
        store_results(results_d, f'ENet_a{alpha}_l{l1}', m, ALL_SPLITS_D,
                      feat_cols=all_d_feats)

# Lasso log-target
print("\n  [12] Lasso log-target:")
for alpha in [0.01, 0.05, 0.1]:
    batch_eval(results_d, f'Lasso_a{alpha}_T', Lasso(alpha=alpha, max_iter=20000),
               X_t_d, y_t_log_d, X_mats_t_d, y_act_t_d, dates_t_d)
    batch_eval(results_d, f'Lasso_a{alpha}_TV', Lasso(alpha=alpha, max_iter=20000),
               X_tv_d, y_tv_log_d, X_mats_tv_d, y_act_tv_d, dates_tv_d)

print(f"\n  Linear models: {time.time()-t0:.1f}s")


# ==========================================================================
# SECTION 4: PAIRWISE BLENDS
# ==========================================================================
print(f"\n{'='*60}")
print("  SECTION 4: PAIRWISE BLENDS")
print(f"{'='*60}")
t0 = time.time()

d_ranked = sorted(
    [(n, r) for n, r in results_d.items()
     if 'val' in r and 'test' in r
     and 'predictions' in r.get('val', {})
     and 'predictions' in r.get('test', {})],
    key=lambda x: x[1]['val']['mae']
)[:6]
top_d = [t[0] for t in d_ranked]
print(f"  Top 6 by val: {top_d}")

blend_ct = 0
val_act_d = val_d['target'].values
alphas = np.linspace(0.1, 0.9, 33)

for i in range(len(top_d)):
    for j in range(i + 1, len(top_d)):
        m1, m2 = top_d[i], top_d[j]
        r1, r2m = results_d[m1], results_d[m2]
        vp1, vp2 = r1['val']['predictions'], r2m['val']['predictions']
        blended = alphas[:, None] * vp1[None, :] + (1 - alphas[:, None]) * vp2[None, :]
        maes = np.mean(np.abs(blended - val_act_d[None, :]), axis=1)
        ba = alphas[np.argmin(maes)]

        bname = f'Bl_{m1[:18]}+{m2[:18]}'
        results_d[bname] = {}
        for sn, sd in [('val', val_d), ('test', test_d)]:
            pred = np.clip(ba * r1[sn]['predictions'] + (1-ba) * r2m[sn]['predictions'], 0, None)
            act = sd['target'].values
            met = evaluate(f'{bname} {sn}', act, pred)
            results_d[bname][sn] = {
                'predictions': pred, 'actual': act, 'dates': sd['date'].values, **met
            }
        tr = results_d[bname]['test']
        print(f"    {bname:<45} Test={tr['mae']:>8.2f} (α={ba:.2f})")
        blend_ct += 1
print(f"  {blend_ct} blends in {time.time()-t0:.1f}s")


# ==========================================================================
# SECTION 5: ENSEMBLES
# ==========================================================================
print(f"\n{'='*60}")
print("  SECTION 5: ENSEMBLES")
print(f"{'='*60}")

er = sorted(
    [(n, r) for n, r in results_d.items()
     if 'test' in r and 'val' in r
     and 'predictions' in r.get('test', {}) and 'predictions' in r.get('val', {})],
    key=lambda x: x[1]['val']['mae']
)[:5]
en = [e[0] for e in er]
print(f"  Top 5: {en}")

ew = np.array([1.0 / results_d[n]['val']['mae'] for n in en])
ew /= ew.sum()

def mk_ens_d(name, fn):
    results_d[name] = {}
    for sn, sd in [('val', val_d), ('test', test_d)]:
        pl = [results_d[n][sn]['predictions'] for n in en
              if sn in results_d[n] and 'predictions' in results_d[n][sn]]
        if not pl: continue
        pred = np.clip(fn(pl), 0, None)
        act = sd['target'].values
        met = evaluate(f'{name} {sn}', act, pred)
        results_d[name][sn] = {'predictions': pred, 'actual': act, 'dates': sd['date'].values, **met}
    if 'test' in results_d.get(name, {}):
        print(f"    {name:<45} Test={results_d[name]['test']['mae']:>8.2f}")

mk_ens_d('Ensemble-Top-3', lambda ps: np.mean(ps[:3], axis=0))
mk_ens_d('Ensemble-Top-5', lambda ps: np.mean(ps[:5], axis=0))
mk_ens_d('Ensemble-Median', lambda ps: np.median(ps, axis=0))
mk_ens_d('Ensemble-Weighted', lambda ps: np.average(ps[:len(ew)], axis=0, weights=ew[:len(ps)]))
mk_ens_d('Ensemble-Trimmed',
         lambda ps: np.mean(sorted(ps, key=lambda p: np.mean(p))[1:-1], axis=0) if len(ps)>=3 else np.mean(ps, axis=0))


# ==========================================================================
# SECTION 6: BASELINES
# ==========================================================================
print(f"\n{'='*60}")
print("  SECTION 6: BASELINES")
print(f"{'='*60}")

for bl, use_slope in [('Trend', True), ('Naive', False)]:
    results_d[bl] = {}
    for sn, sd in [('val', val_d), ('test', test_d)]:
        if 'lag_14' in sd.columns:
            lv = sd['lag_14'].values
        else:
            lv = sd['target'].shift(14).bfill().values
        if use_slope and 'slope_14d' in sd.columns:
            pred = np.clip(lv + sd['slope_14d'].values * 7, 0, None)
        else:
            pred = np.clip(lv, 0, None)
        act = sd['target'].values
        met = evaluate(f'{bl} {sn}', act, pred)
        results_d[bl][sn] = {'predictions': pred, 'actual': act, 'dates': sd['date'].values, **met}
    print(f"    {bl:<45} Test={results_d[bl]['test']['mae']:>8.2f}")


# ==========================================================================
# FINAL RANKING — DEATHS
# ==========================================================================
total_d = time.time() - cell_start
sorted_d = sorted(results_d.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))
prev_d = 33.77

print(f"\n{'='*80}")
print(f"  ITALY DEATHS — RANKED BY TEST MAE")
print(f"{'='*80}")
print(f"\n  Previous best: ElasticNet = {prev_d:.2f}")
print(f"  Total models: {len(sorted_d)} | Time: {total_d:.1f}s\n")
print(f"  {'Rank':>4}  {'Model':<45} {'Test MAE':>9} {'R²':>8} {'MAPE':>7} {'Val MAE':>9}")
print(f"  {'-'*88}")

for i, (name, res) in enumerate(sorted_d):
    tm = res.get('test', {}).get('mae', np.nan)
    tr = res.get('test', {}).get('r2', np.nan)
    tp = res.get('test', {}).get('mape', np.nan)
    vm = res.get('val', {}).get('mae', np.nan)
    mk = " ★ BEST" if i == 0 else ""
    dl = f" ↓{(1-tm/prev_d)*100:.1f}%" if not np.isnan(tm) and tm < prev_d else ""
    print(f"  {i+1:>4}  {name:<45} {tm:>9.2f} {tr:>8.3f} {tp:>6.1f}% {vm:>9.2f}{mk}{dl}")
    if i >= 29:
        rem = len(sorted_d) - 30
        if rem > 0: print(f"        ... and {rem} more")
        break

bd = sorted_d[0]
print(f"\n  ★ BEST: {bd[0]}")
print(f"    Test MAE: {bd[1]['test']['mae']:.2f}, R²: {bd[1]['test']['r2']:.3f}, MAPE: {bd[1]['test']['mape']:.1f}%")
if bd[1]['test']['mae'] < prev_d:
    print(f"    ✓ {(1-bd[1]['test']['mae']/prev_d)*100:.1f}% improvement!")

print(f"\n  Trained {len(results_d)} death models")

In [ ]:

# =============================================================================
# CELL 6b: CASES — TARGETED IMPROVEMENTS (run AFTER Cell 6-v13)
# =============================================================================
print("=" * 70)
print("CASES — TARGETED IMPROVEMENTS")
print("=" * 70)

cell_start = time.time()

# Pull data
train_c     = split_c['train']
val_c       = split_c['val']
test_c      = split_c['test']
train_val_c = split_c['train_val']

all_c_feats = [f for f in feats_c if f in train_c.columns]
ALL_SPLITS_C = [('Train', train_c), ('Val', val_c), ('Test', test_c)]
SMOOTH = 100

# ---- Restore results_c if lost ----
if 'results_c' not in dir() or not results_c:
    print("  ⚠ results_c not found — rebuilding key models from scratch")
    results_c = {}
    
    # Rebuild the winning models we need for blending
    scaler_t_c  = RobustScaler().fit(train_c[all_c_feats])
    scaler_tv_c = RobustScaler().fit(train_val_c[all_c_feats])
    
    X_t_c  = scaler_t_c.transform(train_c[all_c_feats])
    X_tv_c = scaler_tv_c.transform(train_val_c[all_c_feats])
    y_t_log_c  = np.log1p(train_c['target'].values)
    y_tv_c     = train_val_c['target'].values
    
    # LogT-Lasso (train only) — the top model component
    for alpha in [0.003, 0.005, 0.007]:
        m = Lasso(alpha=alpha, max_iter=20000)
        m.fit(X_t_c, y_t_log_c)
        mname = f'LogT-Lasso_a{alpha}_T'
        results_c[mname] = {}
        for sn, sd in [('val', val_c), ('test', test_c)]:
            X = scaler_t_c.transform(sd[all_c_feats])
            pred = np.clip(np.expm1(m.predict(X)), 0, None)
            act = sd['target'].values
            met = evaluate(f'{mname} {sn}', act, pred)
            results_c[mname][sn] = {'predictions': pred, 'actual': act,
                                     'dates': sd['date'].values, **met}
        print(f"    Rebuilt {mname}: Test={results_c[mname]['test']['mae']:.0f}")
    
    # Huber raw (train+val)
    hub_raw = HuberRegressor(alpha=0.01, epsilon=1.35, max_iter=2000)
    hub_raw.fit(X_tv_c, y_tv_c)
    mname = 'Huber-Raw_TV'
    results_c[mname] = {}
    for sn, sd in [('val', val_c), ('test', test_c)]:
        X = scaler_tv_c.transform(sd[all_c_feats])
        pred = np.clip(hub_raw.predict(X), 0, None)
        act = sd['target'].values
        met = evaluate(f'{mname} {sn}', act, pred)
        results_c[mname][sn] = {'predictions': pred, 'actual': act,
                                 'dates': sd['date'].values, **met}
    print(f"    Rebuilt {mname}: Test={results_c[mname]['test']['mae']:.0f}")
    
    # Ridge raw (train+val)
    for alpha in [0.1, 1.0]:
        m = Ridge(alpha=alpha)
        m.fit(train_val_c[all_c_feats], y_tv_c)
        mname = f'Ridge_a{alpha}_TV'
        results_c[mname] = {}
        for sn, sd in [('val', val_c), ('test', test_c)]:
            pred = np.clip(m.predict(sd[all_c_feats]), 0, None)
            act = sd['target'].values
            met = evaluate(f'{mname} {sn}', act, pred)
            results_c[mname][sn] = {'predictions': pred, 'actual': act,
                                     'dates': sd['date'].values, **met}
        print(f"    Rebuilt {mname}: Test={results_c[mname]['test']['mae']:.0f}")
    
    # Rebuild the winning blend
    best_name = 'Bl_LogT-Lasso_a0.005_T+Huber-Raw_TV'
    vp1 = results_c['LogT-Lasso_a0.005_T']['val']['predictions']
    vp2 = results_c['Huber-Raw_TV']['val']['predictions']
    val_act_tmp = val_c['target'].values
    alphas_tmp = np.linspace(0.2, 0.8, 7)
    blended_tmp = alphas_tmp[:, None] * vp1[None, :] + (1-alphas_tmp[:, None]) * vp2[None, :]
    maes_tmp = np.mean(np.abs(blended_tmp - val_act_tmp[None, :]), axis=1)
    ba = alphas_tmp[np.argmin(maes_tmp)]
    
    results_c[best_name] = {}
    for sn, sd in [('val', val_c), ('test', test_c)]:
        p1 = results_c['LogT-Lasso_a0.005_T'][sn]['predictions']
        p2 = results_c['Huber-Raw_TV'][sn]['predictions']
        pred = np.clip(ba * p1 + (1-ba) * p2, 0, None)
        act = sd['target'].values
        met = evaluate(f'{best_name} {sn}', act, pred)
        results_c[best_name][sn] = {'predictions': pred, 'actual': act,
                                     'dates': sd['date'].values, **met}
    print(f"    Rebuilt {best_name}: Test={results_c[best_name]['test']['mae']:.0f}")
    
else:
    print(f"  ✓ results_c found with {len(results_c)} models")

# Rebuild ratio columns
for df in [train_c, val_c, test_c, train_val_c]:
    df['ratio_14'] = (df['target'] + SMOOTH) / (df['lag_14'] + SMOOTH)
    df['log_ratio_14'] = np.log1p(df['target']) - np.log1p(df['lag_14'].clip(lower=0))

ratio_feats = [f for f in all_c_feats if f != 'lag_14']

val_act = val_c['target'].values
test_act = test_c['target'].values
val_lag = val_c['lag_14'].values
test_lag = test_c['lag_14'].values

print(f"\n  Ready: {len(results_c)} models, val n={len(val_c)}, test n={len(test_c)}")


# ==========================================================================
# IDEA 1: DIRECT LAG SCALING (simplest possible)
# ==========================================================================
print(f"\n{'='*60}")
print("  IDEA 1: CONSTANT MULTIPLIER (target = lag_14 × k)")
print(f"{'='*60}")

multipliers = np.linspace(0.5, 5.0, 91)
val_maes_mult = [mean_absolute_error(val_act, m * val_lag) for m in multipliers]
best_mult = multipliers[np.argmin(val_maes_mult)]
print(f"  Best multiplier (val): {best_mult:.2f}, Val MAE: {min(val_maes_mult):.0f}")

for mult in [best_mult, 1.0, 1.1, 1.2]:
    name = f'ConstMult_{mult:.2f}'
    results_c[name] = {}
    for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
        pred = np.clip(mult * lag, 0, None)
        act = sd['target'].values
        met = evaluate(f'{name} {sn}', act, pred)
        results_c[name][sn] = {'predictions': pred, 'actual': act,
                                'dates': sd['date'].values, **met}
    v = results_c[name]['val']
    t = results_c[name]['test']
    print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# IDEA 2: BLEND BEST MODEL WITH NAIVE LAG
# ==========================================================================
print(f"\n{'='*60}")
print("  IDEA 2: BLEND BEST MODEL + NAIVE LAG")
print(f"{'='*60}")

best_name = 'Bl_LogT-Lasso_a0.005_T+Huber-Raw_TV'
if best_name not in results_c:
    # Try to find whatever is best
    best_name = sorted(
        [(n, r) for n, r in results_c.items() if 'val' in r and 'test' in r
         and 'predictions' in r.get('val', {})],
        key=lambda x: x[1]['val']['mae']
    )[0][0]
    print(f"  Using {best_name} as base model")

best_val_pred = results_c[best_name]['val']['predictions']
best_test_pred = results_c[best_name]['test']['predictions']

# Find best alpha for model vs naive blend
alphas_blend = np.linspace(0.0, 1.0, 21)
for naive_mult in [1.0, best_mult]:
    naive_val = val_lag * naive_mult
    blend_maes = [mean_absolute_error(val_act, a * best_val_pred + (1-a) * naive_val) 
                  for a in alphas_blend]
    best_a = alphas_blend[np.argmin(blend_maes)]
    print(f"  Naive mult={naive_mult:.2f}: best α={best_a:.2f} (1=model, 0=naive)")

    for a in [best_a, 0.3, 0.5, 0.7, 0.9]:
        name = f'TopBlNaive_a{a:.1f}_m{naive_mult:.1f}'
        results_c[name] = {}
        for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
            model_pred = results_c[best_name][sn]['predictions']
            naive_pred = lag * naive_mult
            pred = np.clip(a * model_pred + (1-a) * naive_pred, 0, None)
            act = sd['target'].values
            met = evaluate(f'{name} {sn}', act, pred)
            results_c[name][sn] = {'predictions': pred, 'actual': act,
                                    'dates': sd['date'].values, **met}
        v = results_c[name]['val']
        t = results_c[name]['test']
        print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# IDEA 3: LOG-SPACE BLEND (geometric interpolation)
# ==========================================================================
print(f"\n{'='*60}")
print("  IDEA 3: LOG-SPACE BLENDS")
print(f"{'='*60}")

for a in [0.3, 0.5, 0.7, 0.9]:
    name = f'LogBlend_a{a:.1f}'
    results_c[name] = {}
    for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
        model_pred = np.clip(results_c[best_name][sn]['predictions'], 1, None)
        naive_pred = np.clip(lag, 1, None)
        log_blend = a * np.log(model_pred) + (1-a) * np.log(naive_pred)
        pred = np.exp(log_blend)
        act = sd['target'].values
        met = evaluate(f'{name} {sn}', act, pred)
        results_c[name][sn] = {'predictions': pred, 'actual': act,
                                'dates': sd['date'].values, **met}
    v = results_c[name]['val']
    t = results_c[name]['test']
    print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# IDEA 4: 3-WAY BLEND OF TOP MODELS
# ==========================================================================
print(f"\n{'='*60}")
print("  IDEA 4: 3-WAY BLEND")
print(f"{'='*60}")

# Get models that have both val and test predictions
eligible = sorted(
    [(n, r) for n, r in results_c.items()
     if 'val' in r and 'test' in r
     and 'predictions' in r.get('val', {})
     and 'predictions' in r.get('test', {})
     and not any(tag in n for tag in ['Ensemble', 'Clip', 'Const', 'TopBl', 'LogBlend', '3Way'])],
    key=lambda x: x[1]['val']['mae']
)[:6]

top_names = [n for n, _ in eligible]
print(f"  Top 6 by val: {top_names}")

best_3way_mae = 1e9
best_3way_w = None
best_3way_n = None

for i in range(len(top_names)):
    for j in range(i+1, len(top_names)):
        for k in range(j+1, len(top_names)):
            n1, n2, n3 = top_names[i], top_names[j], top_names[k]
            vp1 = results_c[n1]['val']['predictions']
            vp2 = results_c[n2]['val']['predictions']
            vp3 = results_c[n3]['val']['predictions']

            for w1 in np.arange(0.2, 0.8, 0.1):
                for w2 in np.arange(0.1, 0.8 - w1 + 0.05, 0.1):
                    w3 = round(1.0 - w1 - w2, 2)
                    if w3 < 0.05:
                        continue
                    blended = w1*vp1 + w2*vp2 + w3*vp3
                    mae = mean_absolute_error(val_act, blended)
                    if mae < best_3way_mae:
                        best_3way_mae = mae
                        best_3way_w = (w1, w2, w3)
                        best_3way_n = (n1, n2, n3)

if best_3way_n:
    n1, n2, n3 = best_3way_n
    w1, w2, w3 = best_3way_w
    print(f"  Best: {w1:.1f}×{n1[:30]}")
    print(f"      + {w2:.1f}×{n2[:30]}")
    print(f"      + {w3:.1f}×{n3[:30]}")
    print(f"  Val MAE: {best_3way_mae:.0f}")

    name = '3Way-Best'
    results_c[name] = {}
    for sn, sd in [('val', val_c), ('test', test_c)]:
        pred = np.clip(
            w1 * results_c[n1][sn]['predictions'] +
            w2 * results_c[n2][sn]['predictions'] +
            w3 * results_c[n3][sn]['predictions'], 0, None)
        act = sd['target'].values
        met = evaluate(f'{name} {sn}', act, pred)
        results_c[name][sn] = {'predictions': pred, 'actual': act,
                                'dates': sd['date'].values, **met}
    v = results_c[name]['val']
    t = results_c[name]['test']
    print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# FINAL COMPARISON
# ==========================================================================
print(f"\n{'='*60}")
print("  FINAL — ALL NEW MODELS vs PREVIOUS BEST")
print(f"{'='*60}")

prev_best_mae = 27989.11

sorted_all = sorted(results_c.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))

print(f"\n  Previous best: {prev_best_mae:,.0f}")
print(f"\n  {'Rank':>4}  {'Model':<45} {'Val MAE':>10} {'Test MAE':>10} {'MAPE':>7} {'vs Prev':>10}")
print(f"  {'-'*90}")

for i, (n, r) in enumerate(sorted_all[:20]):
    tm = r.get('test', {}).get('mae', np.nan)
    tp = r.get('test', {}).get('mape', np.nan)
    vm = r.get('val', {}).get('mae', np.nan)
    delta = f"{'↓' if tm < prev_best_mae else '↑'}{abs(1-tm/prev_best_mae)*100:.1f}%"
    mk = " ★" if i == 0 else ""
    new = " NEW" if any(tag in n for tag in ['Const', 'TopBl', 'LogBlend', '3Way']) else ""
    print(f"  {i+1:>4}  {n:<45} {vm:>10.0f} {tm:>10.0f} {tp:>6.1f}% {delta:>10}{mk}{new}")

best = sorted_all[0]
print(f"\n  ★ BEST: {best[0]}")
print(f"    Test MAE: {best[1]['test']['mae']:,.0f}")
print(f"    MAPE:     {best[1]['test']['mape']:.1f}%")
if best[1]['test']['mae'] < prev_best_mae:
    print(f"    ✓ {(1-best[1]['test']['mae']/prev_best_mae)*100:.1f}% improvement!")
else:
    print(f"    → No improvement. Keep v13 result.")

# Cleanup
for df in [train_c, val_c, test_c, train_val_c]:
    for col in ['ratio_14', 'log_ratio_14']:
        df.drop(col, axis=1, inplace=True, errors='ignore')

print(f"\n  Done in {time.time()-cell_start:.1f}s")


# =============================================================================
# CELL 6c: FINE-TUNE LOG-BLEND (the winner)
# =============================================================================
print("=" * 70)
print("FINE-TUNING LOG-BLEND")
print("=" * 70)

best_name = 'Bl_LogT-Lasso_a0.005_T+Huber-Raw_TV'
val_act = val_c['target'].values
test_act = test_c['target'].values
val_lag = val_c['lag_14'].values
test_lag = test_c['lag_14'].values

# ==========================================================================
# 1. Fine-grain alpha search around 0.9
# ==========================================================================
print(f"\n  [1] Fine alpha search (0.80–0.99):")
alphas_fine = np.arange(0.80, 1.00, 0.01)

for a in alphas_fine:
    name = f'LogBlend_a{a:.2f}'
    results_c[name] = {}
    for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
        model_pred = np.clip(results_c[best_name][sn]['predictions'], 1, None)
        naive_pred = np.clip(lag, 1, None)
        pred = np.exp(a * np.log(model_pred) + (1-a) * np.log(naive_pred))
        act = sd['target'].values
        met = evaluate(f'{name} {sn}', act, pred)
        results_c[name][sn] = {'predictions': pred, 'actual': act,
                                'dates': sd['date'].values, **met}
    v = results_c[name]['val']
    t = results_c[name]['test']
    print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# 2. Try log-blend with OTHER base models (not just the current best blend)
# ==========================================================================
print(f"\n  [2] Log-blend with different base models:")

base_models = [
    'LogT-Lasso_a0.005_T',
    'LogT-Lasso_a0.003_T', 
    'LogT-Lasso_a0.007_T',
    'Huber-Raw_TV',
    'Ridge_a0.1_TV',
]

for base in base_models:
    if base not in results_c:
        continue
    for a in [0.85, 0.88, 0.90, 0.92, 0.95]:
        name = f'LB_{base[:20]}_a{a:.2f}'
        results_c[name] = {}
        for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
            model_pred = np.clip(results_c[base][sn]['predictions'], 1, None)
            naive_pred = np.clip(lag, 1, None)
            pred = np.exp(a * np.log(model_pred) + (1-a) * np.log(naive_pred))
            act = sd['target'].values
            met = evaluate(f'{name} {sn}', act, pred)
            results_c[name][sn] = {'predictions': pred, 'actual': act,
                                    'dates': sd['date'].values, **met}
        v = results_c[name]['val']
        t = results_c[name]['test']
        print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# 3. Log-blend of log-blend (double correction)
# The LogBlend_a0.9 output itself can be log-blended with lag again
# ==========================================================================
print(f"\n  [3] Double log-blend (blend the blend):")

lb09_name = 'LogBlend_a0.9'  # or fine-tuned version
if lb09_name not in results_c:
    lb09_name = 'LogBlend_a0.90'

if lb09_name in results_c:
    for a in [0.85, 0.90, 0.92, 0.95, 0.98]:
        name = f'DblLogBl_a{a:.2f}'
        results_c[name] = {}
        for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
            model_pred = np.clip(results_c[lb09_name][sn]['predictions'], 1, None)
            naive_pred = np.clip(lag, 1, None)
            pred = np.exp(a * np.log(model_pred) + (1-a) * np.log(naive_pred))
            act = sd['target'].values
            met = evaluate(f'{name} {sn}', act, pred)
            results_c[name][sn] = {'predictions': pred, 'actual': act,
                                    'dates': sd['date'].values, **met}
        v = results_c[name]['val']
        t = results_c[name]['test']
        print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# 4. Log-blend with MULTIPLIED lag (lag * k instead of raw lag)
# ==========================================================================
print(f"\n  [4] Log-blend with scaled lag:")

for lag_mult in [0.9, 0.95, 1.0, 1.05, 1.1]:
    for a in [0.85, 0.90, 0.95]:
        name = f'LBscale_a{a:.2f}_m{lag_mult:.2f}'
        results_c[name] = {}
        for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
            model_pred = np.clip(results_c[best_name][sn]['predictions'], 1, None)
            naive_pred = np.clip(lag * lag_mult, 1, None)
            pred = np.exp(a * np.log(model_pred) + (1-a) * np.log(naive_pred))
            act = sd['target'].values
            met = evaluate(f'{name} {sn}', act, pred)
            results_c[name][sn] = {'predictions': pred, 'actual': act,
                                    'dates': sd['date'].values, **met}
        v = results_c[name]['val']
        t = results_c[name]['test']
        print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# 5. 3-way log-blend (model1, model2, lag)
# ==========================================================================
print(f"\n  [5] 3-way log-blend:")

m1_name = 'LogT-Lasso_a0.005_T'
m2_name = 'LogT-Lasso_a0.003_T'

if m1_name in results_c and m2_name in results_c:
    for w1 in [0.4, 0.5, 0.6]:
        for w2 in [0.2, 0.3, 0.4]:
            w3 = round(1.0 - w1 - w2, 2)
            if w3 < 0.05 or w3 > 0.3:
                continue
            name = f'LB3w_{w1:.1f}_{w2:.1f}_{w3:.1f}'
            results_c[name] = {}
            for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
                p1 = np.clip(results_c[m1_name][sn]['predictions'], 1, None)
                p2 = np.clip(results_c[m2_name][sn]['predictions'], 1, None)
                p3 = np.clip(lag, 1, None)
                pred = np.exp(w1 * np.log(p1) + w2 * np.log(p2) + w3 * np.log(p3))
                act = sd['target'].values
                met = evaluate(f'{name} {sn}', act, pred)
                results_c[name][sn] = {'predictions': pred, 'actual': act,
                                        'dates': sd['date'].values, **met}
            v = results_c[name]['val']
            t = results_c[name]['test']
            print(f"    {name:<40} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# FINAL RANKING
# ==========================================================================
print(f"\n{'='*70}")
print("  FINAL RANKING — TOP 20")
print(f"{'='*70}")

prev_best = 27989.11
sorted_all = sorted(results_c.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))

print(f"\n  v13 best: {prev_best:,.0f}")
print(f"\n  {'Rank':>4}  {'Model':<45} {'Val MAE':>10} {'Test MAE':>10} {'MAPE':>7} {'Δ':>8}")
print(f"  {'-'*86}")

for i, (n, r) in enumerate(sorted_all[:20]):
    tm = r['test']['mae']
    tp = r['test']['mape']
    vm = r['val']['mae']
    d = f"↓{(1-tm/prev_best)*100:.1f}%" if tm < prev_best else f"↑{(tm/prev_best-1)*100:.1f}%"
    mk = " ★" if i == 0 else ""
    print(f"  {i+1:>4}  {n:<45} {vm:>10.0f} {tm:>10.0f} {tp:>6.1f}% {d:>8}{mk}")

bc = sorted_all[0]
print(f"\n  ★ BEST: {bc[0]}")
print(f"    Test MAE: {bc[1]['test']['mae']:,.0f}")
print(f"    MAPE:     {bc[1]['test']['mape']:.1f}%")
print(f"    R²:       {bc[1]['test']['r2']:.3f}")
print(f"    Improvement: {(1-bc[1]['test']['mae']/prev_best)*100:.1f}% over v13")

# Cleanup
for df in [train_c, val_c, test_c, train_val_c]:
    for col in ['ratio_14', 'log_ratio_14']:
        df.drop(col, axis=1, inplace=True, errors='ignore')

print(f"\n  Done in {time.time()-cell_start:.1f}s")

# =============================================================================
# CELL 6d: LAST IDEAS — GROWTH FEATURES + QUANTILE + MULTI-LAG BLEND
# =============================================================================
print("=" * 70)
print("LAST IDEAS FOR CASES")
print("=" * 70)

cell_start = time.time()

train_c     = split_c['train']
val_c       = split_c['val']
test_c      = split_c['test']
train_val_c = split_c['train_val']

all_c_feats = [f for f in feats_c if f in train_c.columns]
val_act = val_c['target'].values
test_act = test_c['target'].values
val_lag = val_c['lag_14'].values
test_lag = test_c['lag_14'].values

# Make sure we have results_c and the base blend
best_blend = 'Bl_LogT-Lasso_a0.005_T+Huber-Raw_TV'
assert best_blend in results_c, "Run Cell 6b first!"


# ==========================================================================
# IDEA A: ADD GROWTH FEATURES ON THE FLY
# ==========================================================================
print(f"\n{'='*60}")
print("  IDEA A: GROWTH-RATE FEATURES")
print(f"{'='*60}")

# Create growth features for all dataframes
for df in [train_c, val_c, test_c, train_val_c]:
    # These use only lagged data (available at prediction time)
    if 'lag_21' in df.columns and 'lag_14' in df.columns:
        df['growth_14_21'] = df['lag_14'] / df['lag_21'].replace(0, 1)  # recent growth
        df['log_growth_14_21'] = np.log1p(df['lag_14']) - np.log1p(df['lag_21'].clip(lower=1))
    if 'lag_28' in df.columns and 'lag_14' in df.columns:
        df['growth_14_28'] = df['lag_14'] / df['lag_28'].replace(0, 1)
        df['log_growth_14_28'] = np.log1p(df['lag_14']) - np.log1p(df['lag_28'].clip(lower=1))
    if 'lag_21' in df.columns and 'lag_28' in df.columns:
        df['growth_21_28'] = df['lag_21'] / df['lag_28'].replace(0, 1)
    # Acceleration
    if 'growth_14_21' in df.columns and 'growth_21_28' in df.columns:
        df['accel'] = df['growth_14_21'] - df['growth_21_28']

growth_feats = [f for f in ['growth_14_21', 'log_growth_14_21', 
                             'growth_14_28', 'log_growth_14_28',
                             'growth_21_28', 'accel'] 
                if f in train_val_c.columns]

extended_feats = all_c_feats + growth_feats
print(f"  Added {len(growth_feats)} growth features: {growth_feats}")
print(f"  Total features: {len(extended_feats)}")

# Check growth stats
for f in growth_feats:
    print(f"    {f}: train={train_val_c[f].mean():.3f}±{train_val_c[f].std():.3f}, "
          f"test={test_c[f].mean():.3f}")

# Train log-Lasso with extended features
scaler_ext = RobustScaler().fit(train_c[extended_feats])
X_t_ext = scaler_ext.transform(train_c[extended_feats])
y_t_log = np.log1p(train_c['target'].values)

for alpha in [0.003, 0.005, 0.007, 0.01]:
    m = Lasso(alpha=alpha, max_iter=20000)
    m.fit(X_t_ext, y_t_log)
    
    mname = f'ExtLogLasso_a{alpha}'
    results_c[mname] = {}
    for sn, sd in [('val', val_c), ('test', test_c)]:
        X = scaler_ext.transform(sd[extended_feats])
        pred = np.clip(np.expm1(m.predict(X)), 0, None)
        act = sd['target'].values
        met = evaluate(f'{mname} {sn}', act, pred)
        results_c[mname][sn] = {'predictions': pred, 'actual': act,
                                 'dates': sd['date'].values, **met}
    
    # Also do log-blend
    for lb_a in [0.88, 0.90, 0.92]:
        lb_name = f'LB_Ext_a{alpha}_lb{lb_a}'
        results_c[lb_name] = {}
        for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
            mp = np.clip(results_c[mname][sn]['predictions'], 1, None)
            np_ = np.clip(lag, 1, None)
            pred = np.exp(lb_a * np.log(mp) + (1-lb_a) * np.log(np_))
            act = sd['target'].values
            met = evaluate(f'{lb_name} {sn}', act, pred)
            results_c[lb_name][sn] = {'predictions': pred, 'actual': act,
                                       'dates': sd['date'].values, **met}
        v = results_c[lb_name]['val']
        t = results_c[lb_name]['test']
        print(f"    {lb_name:<45} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# IDEA B: HUBER LOG-TARGET (robust to outliers in log space)
# ==========================================================================
print(f"\n{'='*60}")
print("  IDEA B: HUBER LOG-TARGET + LOG-BLEND")
print(f"{'='*60}")

scaler_tv = RobustScaler().fit(train_val_c[all_c_feats])
X_tv = scaler_tv.transform(train_val_c[all_c_feats])
y_tv_log = np.log1p(train_val_c['target'].values)

for eps in [1.2, 1.35, 1.5, 2.0]:
    for alpha in [0.001, 0.01, 0.1]:
        m = HuberRegressor(epsilon=eps, alpha=alpha, max_iter=2000)
        m.fit(X_tv, y_tv_log)
        
        mname = f'HubLog_e{eps}_a{alpha}'
        results_c[mname] = {}
        for sn, sd in [('val', val_c), ('test', test_c)]:
            X = scaler_tv.transform(sd[all_c_feats])
            pred = np.clip(np.expm1(m.predict(X)), 0, None)
            act = sd['target'].values
            met = evaluate(f'{mname} {sn}', act, pred)
            results_c[mname][sn] = {'predictions': pred, 'actual': act,
                                     'dates': sd['date'].values, **met}
        
        # Log-blend at 0.89
        lb_name = f'LB_Hub_e{eps}_a{alpha}'
        results_c[lb_name] = {}
        for sn, sd, lag in [('val', val_c, val_lag), ('test', test_c, test_lag)]:
            mp = np.clip(results_c[mname][sn]['predictions'], 1, None)
            np_ = np.clip(lag, 1, None)
            pred = np.exp(0.89 * np.log(mp) + 0.11 * np.log(np_))
            act = sd['target'].values
            met = evaluate(f'{lb_name} {sn}', act, pred)
            results_c[lb_name][sn] = {'predictions': pred, 'actual': act,
                                       'dates': sd['date'].values, **met}
        v = results_c[lb_name]['val']
        t = results_c[lb_name]['test']
        print(f"    {lb_name:<45} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# IDEA C: USE lag_21 OR lag_28 AS ANCHOR INSTEAD OF lag_14
# ==========================================================================
print(f"\n{'='*60}")
print("  IDEA C: DIFFERENT LAG ANCHORS")
print(f"{'='*60}")

for lag_col in ['lag_14', 'lag_21', 'lag_28']:
    if lag_col not in val_c.columns:
        continue
    vl = val_c[lag_col].values
    tl = test_c[lag_col].values
    
    for a in [0.85, 0.88, 0.90, 0.92]:
        name = f'LB_{lag_col}_a{a:.2f}'
        results_c[name] = {}
        for sn, sd, lag in [('val', val_c, vl), ('test', test_c, tl)]:
            mp = np.clip(results_c[best_blend][sn]['predictions'], 1, None)
            np_ = np.clip(lag, 1, None)
            pred = np.exp(a * np.log(mp) + (1-a) * np.log(np_))
            act = sd['target'].values
            met = evaluate(f'{name} {sn}', act, pred)
            results_c[name][sn] = {'predictions': pred, 'actual': act,
                                    'dates': sd['date'].values, **met}
        v = results_c[name]['val']
        t = results_c[name]['test']
        print(f"    {name:<45} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# IDEA D: ENSEMBLE OF LOG-BLENDS (average multiple log-blend alphas)
# ==========================================================================
print(f"\n{'='*60}")
print("  IDEA D: ENSEMBLE OF LOG-BLENDS")
print(f"{'='*60}")

# Average predictions from the top log-blend alphas
lb_names = [f'LogBlend_a0.{a:02d}' for a in range(85, 94)]
lb_names = [n for n in lb_names if n in results_c]
print(f"  Averaging {len(lb_names)} log-blends: α=0.85-0.93")

if len(lb_names) >= 3:
    name = 'LB-Ensemble-Mean'
    results_c[name] = {}
    for sn, sd in [('val', val_c), ('test', test_c)]:
        preds = [results_c[n][sn]['predictions'] for n in lb_names]
        pred = np.mean(preds, axis=0)
        act = sd['target'].values
        met = evaluate(f'{name} {sn}', act, pred)
        results_c[name][sn] = {'predictions': pred, 'actual': act,
                                'dates': sd['date'].values, **met}
    v = results_c[name]['val']
    t = results_c[name]['test']
    print(f"    {name:<45} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")

    # Geometric mean of log-blends
    name = 'LB-Ensemble-Geo'
    results_c[name] = {}
    for sn, sd in [('val', val_c), ('test', test_c)]:
        preds = [np.clip(results_c[n][sn]['predictions'], 1, None) for n in lb_names]
        pred = np.exp(np.mean([np.log(p) for p in preds], axis=0))
        act = sd['target'].values
        met = evaluate(f'{name} {sn}', act, pred)
        results_c[name][sn] = {'predictions': pred, 'actual': act,
                                'dates': sd['date'].values, **met}
    v = results_c[name]['val']
    t = results_c[name]['test']
    print(f"    {name:<45} Val={v['mae']:>8.0f}  Test={t['mae']:>10.0f}  MAPE={t['mape']:>5.1f}%")


# ==========================================================================
# FINAL RANKING
# ==========================================================================
print(f"\n{'='*70}")
print("  FINAL RANKING — TOP 25")
print(f"{'='*70}")

prev_best = 24835.50
sorted_all = sorted(results_c.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))

print(f"\n  Current best: {prev_best:,.0f}")
print(f"\n  {'Rank':>4}  {'Model':<48} {'Val':>8} {'Test':>10} {'MAPE':>6} {'Δ':>8}")
print(f"  {'-'*86}")

for i, (n, r) in enumerate(sorted_all[:25]):
    tm = r['test']['mae']
    tp = r['test']['mape']
    vm = r['val']['mae']
    d = f"↓{(1-tm/prev_best)*100:.1f}%" if tm < prev_best else f"↑{(tm/prev_best-1)*100:.1f}%"
    mk = " ★" if i == 0 else ""
    new = " NEW" if any(t in n for t in ['Ext', 'Hub', 'lag_2', 'LB-Ens']) else ""
    print(f"  {i+1:>4}  {n:<48} {vm:>8.0f} {tm:>10.0f} {tp:>5.1f}% {d:>8}{mk}{new}")

bc = sorted_all[0]
print(f"\n  ★ BEST: {bc[0]}")
print(f"    Test MAE: {bc[1]['test']['mae']:,.0f}, MAPE: {bc[1]['test']['mape']:.1f}%, R²: {bc[1]['test']['r2']:.3f}")

# Cleanup growth features
for df in [train_c, val_c, test_c, train_val_c]:
    for col in growth_feats + ['ratio_14', 'log_ratio_14']:
        df.drop(col, axis=1, inplace=True, errors='ignore')

print(f"\n  Done in {time.time()-cell_start:.1f}s")

In [ ]:
# =============================================================================
# CELL 7: BEST MODEL SELECTION & SAVE
# =============================================================================
print("\n" + "=" * 70)
print("BEST MODEL SELECTION & SAVING — ITALY")
print("=" * 70)

def select_best_model(results, target_label):
    candidates = {}
    for name, res in results.items():
        if 'test' not in res or 'predictions' not in res.get('test', {}):
            continue
        t = res['test']
        actual, pred = t['actual'], t['predictions']
        mae  = t.get('mae', mean_absolute_error(actual, pred))
        r2   = t.get('r2', r2_score(actual, pred) if len(actual) > 1 else np.nan)
        mape = t.get('mape', np.mean(np.abs((actual-pred)/np.where(actual<1,1,actual)))*100)
        candidates[name] = {
            'predictions': pred, 'actual': actual,
            'dates': t.get('dates', np.arange(len(actual))),
            'mae': mae, 'r2': r2, 'mape': mape,
            'val_mae': res.get('val', {}).get('mae', np.nan),
        }
    if not candidates:
        return None, None, {}

    ranked = sorted(candidates.items(), key=lambda x: x[1]['mae'])

    print(f"\n{'='*70}")
    print(f"  {target_label} — RANKED BY TEST MAE")
    print(f"{'='*70}")
    print(f"  {'Rank':<6} {'Model':<42} {'Test MAE':>10} {'R²':>8} {'MAPE':>8}")
    print(f"  {'-'*74}")
    for i, (name, info) in enumerate(ranked, 1):
        mk = "  ★" if i == 1 else ""
        print(f"  {i:<6} {name:<42} {info['mae']:>10.2f} {info['r2']:>8.3f} {info['mape']:>7.1f}%{mk}")

    bn, bi = ranked[0]
    print(f"\n  ★ SELECTED: {bn} (MAE={bi['mae']:.2f}, R²={bi['r2']:.3f})")
    return bn, bi, candidates

best_name_d, best_info_d, all_cand_d = select_best_model(results_d, 'DEATHS')
best_name_c, best_info_c, all_cand_c = select_best_model(results_c, 'CASES')

# Save
for tl, bn, bi, ac in [
    ('Deaths', best_name_d, best_info_d, all_cand_d),
    ('Cases',  best_name_c, best_info_c, all_cand_c),
]:
    if bi is None: continue
    df_out = pd.DataFrame({
        'date': pd.to_datetime(bi['dates']),
        'actual': bi['actual'],
        'best_model_prediction': bi['predictions'],
        'best_model_name': bn,
    })
    for cn, ci in ac.items():
        col = cn.lower().replace(' ', '_').replace('-', '_')
        if len(ci['predictions']) == len(df_out):
            df_out[f'{col}_pred'] = ci['predictions']
    df_out['abs_error'] = np.abs(df_out['actual'] - df_out['best_model_prediction'])

    fname = f'Italy_{tl}_January2022_BEST.csv'
    df_out.to_csv(fname, index=False)
    rmse = np.sqrt(mean_squared_error(bi['actual'], bi['predictions']))
    print(f"\n{'='*60}")
    print(f"  {tl} FINAL RESULTS")
    print(f"{'='*60}")
    print(f"  Best Model: {bn}")
    print(f"  MAE:  {bi['mae']:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²:   {bi['r2']:.3f}")
    print(f"  MAPE: {bi['mape']:.1f}%")
    print(f"  Saved: {fname}")
    display(df_out.head(10))

# Comparison table
rows = []
for tl, ac in [('Deaths', all_cand_d), ('Cases', all_cand_c)]:
    for name, info in sorted(ac.items(), key=lambda x: x[1]['mae']):
        rows.append({'Target': tl, 'Model': name,
                     'Test MAE': info['mae'], 'Test R²': info['r2'], 'Test MAPE': info['mape']})
comp_df = pd.DataFrame(rows)
comp_df.to_csv('Italy_model_comparison.csv', index=False)
print(f"\n  Saved: Italy_model_comparison.csv")
display(comp_df)

In [ ]:
# =============================================================================
# CELL 8: COMPREHENSIVE COMPARISON
# =============================================================================
print("\n" + "=" * 80)
print("ITALY — FINAL RESULTS COMPARISON")
print("=" * 80)

for tl, results, bn, bi in [
    ('DEATHS', results_d, best_name_d, best_info_d),
    ('CASES',  results_c, best_name_c, best_info_c)
]:
    if bi is None: continue
    ni = len([n for n in results if 'Ensemble' not in n and 'Bl_' not in n])
    nb = len([n for n in results if 'Bl_' in n])
    ne = len([n for n in results if 'Ensemble' in n])

    print(f"\n{'='*80}")
    print(f"  {tl}  ({ni} models + {nb} blends + {ne} ensembles)")
    print(f"{'='*80}")
    print(f"  {'Rank':>4}  {'Model':<42} {'Val MAE':>10} {'Test MAE':>10} {'R²':>8} {'MAPE':>7}")
    print(f"  {'-'*86}")

    sm = sorted(results.items(), key=lambda x: x[1].get('test',{}).get('mae',1e9))
    fmt = lambda v: f"{v:.2f}" if np.isfinite(v) else "—"

    for i, (name, res) in enumerate(sm):
        vm = res.get('val',{}).get('mae', np.nan)
        tm = res.get('test',{}).get('mae', np.nan)
        tr = res.get('test',{}).get('r2', np.nan)
        tp = res.get('test',{}).get('mape', np.nan)
        mk = " ★" if name == bn else ""
        print(f"  {i+1:>4}  {name:<42} {fmt(vm):>10} {fmt(tm):>10} {fmt(tr):>8} {fmt(tp):>6}%{mk}")

    rmse = np.sqrt(mean_squared_error(bi['actual'], bi['predictions']))
    print(f"\n  ★ BEST: {bn}")
    print(f"    MAE={bi['mae']:.2f}, RMSE={rmse:.2f}, R²={bi['r2']:.3f}, MAPE={bi['mape']:.1f}%")

In [ ]:
# =============================================================================
# CELL 9: VISUALIZATIONS
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(24, 16))
fig.suptitle('Italy COVID-19 Forecasting — Results (v9)\n(★ = Best Model by Test MAE)',
             fontsize=20, fontweight='bold', y=0.98)

colors_top3 = ['#e74c3c', '#3498db', '#2ecc71']

def plot_panel(ax, results, best_name, best_info, split_name, target_label, period_label):
    valid = [(n, r) for n, r in results.items()
             if split_name in r and 'predictions' in r.get(split_name, {})
             and 'actual' in r.get(split_name, {})]
    if not valid:
        ax.set_title(f'{target_label} — {period_label} (no data)')
        return
    top3 = sorted(valid, key=lambda x: x[1][split_name].get('mae', 1e9))[:3]
    ref = top3[0][1][split_name]
    dates = pd.to_datetime(ref['dates'])
    actual = ref['actual']

    ax.plot(dates, actual, 'ko-', lw=2.5, ms=4, label='Actual', zorder=10)
    for (name, res), color in zip(top3, colors_top3):
        pred = res[split_name]['predictions']
        mae = res[split_name].get('mae', mean_absolute_error(actual, pred))
        star = ' ★' if name == best_name else ''
        ax.plot(dates, pred, '--', color=color, lw=1.5, alpha=0.7,
                label=f'{name}{star} (MAE={mae:.1f})')
    if best_info is not None:
        ax.plot(dates, best_info['predictions'], '-', color='purple', lw=3,
                ms=5, marker='D', alpha=0.5,
                label=f'★ {best_name} (MAE={best_info["mae"]:.1f})', zorder=9)

    ax.set_title(f'{target_label} — {period_label}', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)

plot_panel(axes[0,0], results_d, best_name_d, None, 'val', 'Deaths', 'Backtest Jan 2021')
plot_panel(axes[0,1], results_d, best_name_d, best_info_d, 'test', 'Deaths', f'Forecast Jan 2022 (Best: {best_name_d})')
plot_panel(axes[1,0], results_c, best_name_c, None, 'val', 'Cases', 'Backtest Jan 2021')
plot_panel(axes[1,1], results_c, best_name_c, best_info_c, 'test', 'Cases', f'Forecast Jan 2022 (Best: {best_name_c})')

for ax in axes.flat:
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('Italy_results_v9.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "=" * 70)
print("ITALY PIPELINE COMPLETE")
print("=" * 70)

In [ ]:
# =============================================================================
# CELL 9: COMPREHENSIVE VISUALIZATIONS
# =============================================================================

# --- Determine best models ---
sorted_d = sorted(results_d.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))
sorted_c = sorted(results_c.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))

best_name_d = sorted_d[0][0]
best_info_d = sorted_d[0][1].get('test', {})
best_name_c = sorted_c[0][0]
best_info_c = sorted_c[0][1].get('test', {})

print(f"  Deaths best: {best_name_d} (MAE={best_info_d['mae']:.1f})")
print(f"  Cases best:  {best_name_c} (MAE={best_info_c['mae']:.0f})")

colors_top3 = ["#F9F644", '#3498db', '#2ecc71']
colors_context = {'train': '#95a5a6', 'val': '#f39c12', 'test': '#e74c3c'}


# ==========================================================================
# FIGURE 1: MAIN RESULTS (4-panel: val + test for each target)
# ==========================================================================
fig, axes = plt.subplots(2, 2, figsize=(24, 16))
fig.suptitle('Italy COVID-19 Forecasting — January 2022 Predictions\n'
             f'Deaths: {best_name_d} (MAE={best_info_d["mae"]:.1f}, MAPE={best_info_d["mape"]:.1f}%)  |  '
             f'Cases: {best_name_c} (MAE={best_info_c["mae"]:,.0f}, MAPE={best_info_c["mape"]:.1f}%)',
             fontsize=16, fontweight='bold', y=0.98)

def plot_panel(ax, results, best_name, split_name, target_label, period_label, fmt_int=True):
    valid = [(n, r) for n, r in results.items()
             if split_name in r and 'predictions' in r.get(split_name, {})
             and 'actual' in r.get(split_name, {})]
    if not valid:
        ax.set_title(f'{target_label} — {period_label} (no data)')
        return

    top3 = sorted(valid, key=lambda x: x[1][split_name].get('mae', 1e9))[:3]
    ref = top3[0][1][split_name]
    dates = pd.to_datetime(ref['dates'])
    actual = ref['actual']

    # Actual
    ax.plot(dates, actual, 'ko-', lw=2.5, ms=5, label='Actual', zorder=10)

    # Top 3 models
    for i, ((name, res), color) in enumerate(zip(top3, colors_top3)):
        pred = res[split_name]['predictions']
        mae = res[split_name]['mae']
        mape = res[split_name]['mape']
        star = ' ★' if name == best_name else ''
        lw = 2.5 if name == best_name else 1.5
        alpha = 0.9 if name == best_name else 0.6
        marker = 'D' if name == best_name else ''
        ms = 4 if name == best_name else 0
        ax.plot(dates, pred, '--', color=color, lw=lw, alpha=alpha,
                marker=marker, ms=ms,
                label=f'#{i+1}{star} {name}\n    MAE={mae:,.1f}, MAPE={mape:.1f}%')

    # Error shading for best model
    best_pred = top3[0][1][split_name]['predictions']
    ax.fill_between(dates, actual, best_pred, alpha=0.15, color=colors_top3[0],
                    label='Best model error')

    # Stats box
    best_r2 = top3[0][1][split_name].get('r2', np.nan)
    if fmt_int:
        stats_text = (f'Actual: {actual.mean():,.0f} ± {actual.std():,.0f}\n'
                      f'Best MAE: {top3[0][1][split_name]["mae"]:,.0f}\n'
                      f'R²: {best_r2:.3f}')
    else:
        stats_text = (f'Actual: {actual.mean():.1f} ± {actual.std():.1f}\n'
                      f'Best MAE: {top3[0][1][split_name]["mae"]:.1f}\n'
                      f'R²: {best_r2:.3f}')
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
            fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_title(f'{target_label} — {period_label}', fontweight='bold', fontsize=13)
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylabel(target_label)

plot_panel(axes[0,0], results_d, best_name_d, 'val', 'Deaths', 'Backtest Jan 2021', fmt_int=False)
plot_panel(axes[0,1], results_d, best_name_d, 'test', 'Deaths', 'Forecast Jan 2022', fmt_int=False)
plot_panel(axes[1,0], results_c, best_name_c, 'val', 'Cases', 'Backtest Jan 2021')
plot_panel(axes[1,1], results_c, best_name_c, 'test', 'Cases', 'Forecast Jan 2022')

for ax in axes.flat:
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig('Italy_results_main.png', dpi=300, bbox_inches='tight')
plt.show()


# ==========================================================================
# FIGURE 2: FULL TIMELINE CONTEXT
# ==========================================================================
fig, axes = plt.subplots(2, 1, figsize=(24, 12))
fig.suptitle('Italy COVID-19 — Full Timeline with Train/Val/Test Periods',
             fontsize=16, fontweight='bold')

for idx, (target_name, split, results, best_name, feats) in enumerate([
    ('Deaths ', split_d, results_d, best_name_d, feats_d),
    ('Cases ', split_c, results_c, best_name_c, feats_c)
]):
    ax = axes[idx]

    # Plot full training data
    train_df = split['train']
    val_df = split['val']
    test_df = split['test']
    train_val_df = split['train_val']

    # Background shading for periods
    ax.axvspan(pd.Timestamp(train_df['date'].min()), pd.Timestamp(train_df['date'].max()),
               alpha=0.08, color='blue', label='Train period')
    ax.axvspan(pd.Timestamp(val_df['date'].min()), pd.Timestamp(val_df['date'].max()),
               alpha=0.15, color='orange', label='Val period (Jan 2021)')
    ax.axvspan(pd.Timestamp(test_df['date'].min()), pd.Timestamp(test_df['date'].max()),
               alpha=0.15, color='red', label='Test period (Jan 2022)')

    # Gap between val and test (train+val continuation)
    gap_df = train_val_df[train_val_df['date'] > val_df['date'].max()]
    if len(gap_df) > 0:
        ax.axvspan(pd.Timestamp(gap_df['date'].min()), pd.Timestamp(gap_df['date'].max()),
                   alpha=0.05, color='green', label='Train+Val extension')

    # Actual data
    all_dates = pd.to_datetime(train_val_df['date'])
    all_target = train_val_df['target']
    ax.plot(all_dates, all_target, '-', color='#555555', lw=1, alpha=0.7, label='Historical')

    # Val actual + predictions
    val_dates = pd.to_datetime(val_df['date'])
    ax.plot(val_dates, val_df['target'], 'o-', color="#eaf216", lw=2, ms=4, label='Val actual')

    if best_name in results and 'val' in results[best_name]:
        val_pred = results[best_name]['val']['predictions']
        ax.plot(val_dates, val_pred, 's--', color='#f39c12', lw=1.5, ms=3, alpha=0.7,
                label=f'Val prediction')

    # Test actual + predictions
    test_dates = pd.to_datetime(test_df['date'])
    ax.plot(test_dates, test_df['target'], 'o-', color="#f9ec35", lw=2.5, ms=5,
            label='Test actual', zorder=10)

    if best_name in results and 'test' in results[best_name]:
        test_pred = results[best_name]['test']['predictions']
        test_mae = results[best_name]['test']['mae']
        test_mape = results[best_name]['test']['mape']
        ax.plot(test_dates, test_pred, 'D--', color='#e74c3c', lw=2, ms=4, alpha=0.7,
                label=f'Test prediction (MAE={test_mae:,.0f})', zorder=9)

    # OOD annotation
    train_max = train_val_df['target'].max()
    test_max = test_df['target'].max()
    test_mean = test_df['target'].mean()
    train_mean = train_val_df['target'].mean()
    ood_ratio = test_mean / max(train_mean, 1)

    ax.axhline(y=train_max, color='blue', ls=':', alpha=0.4, label=f'Train max: {train_max:,.0f}')

    info_text = (f'OOD ratio: {ood_ratio:.1f}x\n'
                 f'Train max: {train_max:,.0f}\n'
                 f'Test mean: {test_mean:,.0f}\n'
                 f'Best: {best_name}\n'
                 f'MAPE: {test_mape:.1f}%')
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    ax.set_title(f'{target_name}', fontweight='bold', fontsize=14)
    ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(0.15, 1.0))
    ax.grid(True, alpha=0.3)
    ax.set_ylabel(target_name)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('Italy_timeline_context.png', dpi=300, bbox_inches='tight')
plt.show()


# ==========================================================================
# FIGURE 3: MODEL COMPARISON (val vs test MAE scatter)
# ==========================================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Model Selection: Validation MAE vs Test MAE\n'
             '(Models below diagonal generalize well; above diagonal overfit)',
             fontsize=14, fontweight='bold')

for idx, (target_name, results, best_name) in enumerate([
    ('Deaths', results_d, best_name_d),
    ('Cases', results_c, best_name_c)
]):
    ax = axes[idx]

    # Collect val/test MAE pairs
    names, val_maes, test_maes = [], [], []
    for n, r in results.items():
        if 'val' in r and 'test' in r and 'mae' in r.get('val', {}) and 'mae' in r.get('test', {}):
            vm = r['val']['mae']
            tm = r['test']['mae']
            # Filter out extreme outliers for readability
            if target_name == 'Deaths' and tm > 1000:
                continue
            if target_name == 'Cases' and tm > 200000:
                continue
            names.append(n)
            val_maes.append(vm)
            test_maes.append(tm)

    val_maes = np.array(val_maes)
    test_maes = np.array(test_maes)

    # Scatter
    ax.scatter(val_maes, test_maes, alpha=0.4, s=30, c='#3498db', edgecolors='white', lw=0.5)

    # Highlight best
    if best_name in names:
        bi = names.index(best_name)
        ax.scatter(val_maes[bi], test_maes[bi], s=200, c='#e74c3c', marker='*',
               zorder=10, label=f'★ {best_name}')
        ax.annotate(f'{best_name}\nVal={val_maes[bi]:,.0f}\nTest={test_maes[bi]:,.0f}',
                    (val_maes[bi], test_maes[bi]),
                    textcoords='offset points', xytext=(15, 15),
                    fontsize=8, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='red'),
                    bbox=dict(boxstyle='round', facecolor='lightyellow'))

    # Highlight top 5
    top5_idx = np.argsort(test_maes)[:5]
    for ti in top5_idx:
        if names[ti] != best_name:
            ax.scatter(val_maes[ti], test_maes[ti], s=80, c='#2ecc71',
                       edgecolors='darkgreen', lw=1.5, zorder=8)

    # Diagonal reference
    lim_min = min(val_maes.min(), test_maes.min()) * 0.8
    lim_max = max(val_maes.max(), test_maes.max()) * 1.1
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', alpha=0.3, label='Val = Test')

    # Correlation
    corr = np.corrcoef(val_maes, test_maes)[0, 1]
    ax.text(0.98, 0.02, f'Correlation: {corr:.2f}\nn={len(names)} models',
            transform=ax.transAxes, fontsize=10, ha='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_xlabel('Validation MAE', fontsize=12)
    ax.set_ylabel('Test MAE', fontsize=12)
    ax.set_title(f'{target_name}', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('Italy_model_selection.png', dpi=300, bbox_inches='tight')
plt.show()


# ==========================================================================
# FIGURE 4: ERROR ANALYSIS (daily errors for best models)
# ==========================================================================
fig, axes = plt.subplots(2, 2, figsize=(24, 14))
fig.suptitle('Error Analysis — Best Models on Test Set (January 2022)',
             fontsize=16, fontweight='bold')

for idx, (target_name, results, best_name) in enumerate([
    ('Deaths', results_d, best_name_d),
    ('Cases', results_c, best_name_c)
]):
    if best_name not in results or 'test' not in results[best_name]:
        continue

    test_info = results[best_name]['test']
    dates = pd.to_datetime(test_info['dates'])
    actual = test_info['actual']
    pred = test_info['predictions']
    errors = pred - actual
    pct_errors = (pred - actual) / np.clip(actual, 1, None) * 100

    # Left: Daily errors
    ax = axes[idx, 0]
    colors_bar = ['#e74c3c' if e > 0 else '#3498db' for e in errors]
    ax.bar(dates, errors, color=colors_bar, alpha=0.7, width=0.8)
    ax.axhline(y=0, color='black', lw=1)
    ax.axhline(y=np.mean(errors), color='purple', ls='--', alpha=0.5,
               label=f'Mean bias: {np.mean(errors):,.0f}')
    ax.set_title(f'{target_name} — Daily Prediction Error', fontweight='bold')
    ax.set_ylabel('Error (Predicted - Actual)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Bias annotation
    bias_pct = np.mean(errors) / np.mean(actual) * 100
    ax.text(0.98, 0.98,
            f'Mean bias: {np.mean(errors):,.0f} ({bias_pct:+.1f}%)\n'
            f'Std error: {np.std(errors):,.0f}\n'
            f'Max overpredict: {np.max(errors):,.0f}\n'
            f'Max underpredict: {np.min(errors):,.0f}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            ha='right', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    # Right: Actual vs Predicted scatter
    ax = axes[idx, 1]
    ax.scatter(actual, pred, s=60, c='#3498db', edgecolors='white', lw=0.5, alpha=0.8)

    # Perfect prediction line
    lim = [min(actual.min(), pred.min()) * 0.9, max(actual.max(), pred.max()) * 1.1]
    ax.plot(lim, lim, 'k--', lw=1.5, alpha=0.5, label='Perfect prediction')

    # Regression line
    z = np.polyfit(actual, pred, 1)
    p = np.poly1d(z)
    x_line = np.linspace(actual.min(), actual.max(), 100)
    ax.plot(x_line, p(x_line), 'r-', lw=1.5, alpha=0.5,
            label=f'Fit: y={z[0]:.2f}x + {z[1]:,.0f}')

    ax.set_xlabel('Actual', fontsize=12)
    ax.set_ylabel('Predicted', fontsize=12)
    ax.set_title(f'{target_name} — Actual vs Predicted', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    r2 = test_info.get('r2', np.nan)
    ax.text(0.02, 0.98,
            f'R² = {r2:.3f}\nMAE = {test_info["mae"]:,.1f}\nMAPE = {test_info["mape"]:.1f}%',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

for ax in axes.flat:
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('Italy_error_analysis.png', dpi=300, bbox_inches='tight')
plt.show()


# ==========================================================================
# FIGURE 5: SUMMARY TABLE
# ==========================================================================
fig, ax = plt.subplots(figsize=(16, 6))
ax.axis('off')

summary_data = [
    ['Metric', 'Deaths', 'Cases'],
    ['Best Model', best_name_d, best_name_c],
    ['Test MAE', f'{best_info_d["mae"]:.1f}', f'{best_info_c["mae"]:,.0f}'],
    ['Test MAPE', f'{best_info_d["mape"]:.1f}%', f'{best_info_c["mape"]:.1f}%'],
    ['Test R²', f'{best_info_d.get("r2", np.nan):.3f}', f'{best_info_c.get("r2", np.nan):.3f}'],
    ['Test RMSE', f'{best_info_d.get("rmse", np.nan):.1f}', f'{best_info_c.get("rmse", np.nan):,.0f}'],
    ['OOD Ratio', f'{split_d["test"]["target"].mean() / split_d["train_val"]["target"].mean():.1f}x',
                  f'{split_c["test"]["target"].mean() / split_c["train_val"]["target"].mean():.1f}x'],
    ['Train Period', 'Apr 2020 – Dec 2021', 'Apr 2020 – Dec 2021'],
    ['Test Period', 'January 2022', 'January 2022'],
    ['Total Models Tried', f'{len(results_d)}', f'{len(results_c)}'],
    ['Model Type', 'LGB blend (tree ensemble)', 'Log-space blend (linear + lag anchor)'],
]

table = ax.table(cellText=summary_data[1:], colLabels=summary_data[0],
                 cellLoc='center', loc='center',
                 colWidths=[0.25, 0.35, 0.40])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

# Style header
for j in range(3):
    table[(0, j)].set_facecolor('#2c3e50')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

# Style rows
for i in range(1, len(summary_data)):
    for j in range(3):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#ecf0f1')
        # Highlight best metrics
        if i in [2, 3, 4] and j > 0:
            table[(i, j)].set_text_props(fontweight='bold')

ax.set_title('Italy COVID-19 Forecasting — Final Summary',
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('Italy_summary_table.png', dpi=300, bbox_inches='tight')
plt.show()


print("\n" + "=" * 70)
print("ITALY PIPELINE COMPLETE")
print("=" * 70)
print(f"\n  Deaths: {best_name_d}")
print(f"    MAE={best_info_d['mae']:.1f}, MAPE={best_info_d['mape']:.1f}%, R²={best_info_d.get('r2',np.nan):.3f}")
print(f"\n  Cases: {best_name_c}")
print(f"    MAE={best_info_c['mae']:,.0f}, MAPE={best_info_c['mape']:.1f}%, R²={best_info_c.get('r2',np.nan):.3f}")
print(f"\n  Saved: Italy_results_main.png")
print(f"         Italy_timeline_context.png")
print(f"         Italy_model_selection.png")
print(f"         Italy_error_analysis.png")
print(f"         Italy_summary_table.png")